In [2]:
import numpy as np

a = np.array([1,2,3])
b = np.array([0.1,0.2,0.3])
c = np.array([-1,-0.5,0.])
d = np.array([0,0,0])

identity = np.eye(3)
print(identity)

print(a.dot(identity))
print(b.dot(identity))
print(c.dot(identity))
print(d.dot(identity))

this = np.array([2,4,6])
movie = np.array([10,10,10])
rocks = np.array([1,1,1])

print(this + movie + rocks)
print((this.dot(identity) + movie).dot(identity) + rocks)

[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
[1. 2. 3.]
[0.1 0.2 0.3]
[-1.  -0.5  0. ]
[0. 0. 0.]
[13 15 17]
[13. 15. 17.]


In [3]:
def softmax(x):
  x = np.atleast_2d(x)
  temp = np.exp(x)
  return temp / np.sum(temp, axis=1, keepdims=True)

word_vecs = {}
word_vecs['yankees'] = np.array([[0.0,0.0,0.1]])
word_vecs['bears'] = np.array([[0.0,0.0,0.1]])
word_vecs['braves'] = np.array([[0.0,0.0,0.1]])
word_vecs['red'] = np.array([[0.0,0.0,0.0]])
word_vecs['sox'] = np.array([[0.0,0.0,0.0]])
word_vecs['lose'] = np.array([[0.0,0.0,0.0]])
word_vecs['defeat'] = np.array([[0.0,0.0,0.0]])
word_vecs['beat'] = np.array([[0.0,0.0,0.0]])
word_vecs['tie'] = np.array([[0.0,0.0,0.0]])

sent2output = np.random.rand(3, len(word_vecs))

identity = np.eye(3)

In [4]:
layer_0 = word_vecs["red"]
layer_1 = layer_0.dot(identity) + word_vecs['sox']
layer_2 = layer_1.dot(identity) + word_vecs['defeat']

pred = softmax(layer_2.dot(sent2output))
print(pred)

[[0.11111111 0.11111111 0.11111111 0.11111111 0.11111111 0.11111111
  0.11111111 0.11111111 0.11111111]]


In [5]:
Y = np.array([1,0,0,0,0,0,0,0,0])

pred_delta = pred - Y
layer_2_delta = pred_delta.dot(sent2output.T)
defeat_delta = layer_2_delta * 1
layer_1_delta = layer_2_delta.dot(identity.T)
sox_delta = layer_1_delta * 1
layer_0_delta = layer_1_delta.dot(identity.T)

alpha = 0.01

word_vecs['red'] -= layer_0_delta * alpha
word_vecs['sox'] -= sox_delta * alpha
word_vecs['defeat'] -= defeat_delta * alpha

identity -= np.dot(layer_0.reshape(-1,1), layer_1_delta.reshape(-1,1).T) * alpha
identity -= np.dot(layer_1.reshape(-1,1), layer_2_delta.reshape(-1,1).T) * alpha
sent2output -= np.dot(layer_2.reshape(-1,1), pred_delta.reshape(-1,1).T) * alpha

In [6]:
import sys, random, math
from collections import Counter
import numpy as np
import os
import kagglehub

In [7]:
path = kagglehub.dataset_download("roblexnana/the-babi-tasks-for-nlp-qa-system")
file_path = os.path.join(path, "tasks_1-20_v1-2", "en","qa1_single-supporting-fact_train.txt")

f = open(file_path, 'r')
raw = f.readlines()
f.close()

100%|██████████| 16.7M/16.7M [00:00<00:00, 128MB/s]

Extracting files...


In [8]:
tokens = list()
for line in raw[0:1000]:
  tokens.append(line.lower().replace("\n","").split(" ")[1:])

print(tokens[0:3])

[['mary', 'moved', 'to', 'the', 'bathroom.'], ['john', 'went', 'to', 'the', 'hallway.'], ['where', 'is', 'mary?', '\tbathroom\t1']]


In [9]:
vocab = set()
for sent in tokens:
  for word in sent:
    vocab.add(word)

vocab = list(vocab)

word2index = {}
for i, word in enumerate(vocab):
  word2index[word] = i

In [10]:
def word2indices(sentence):
  idx = list()
  for word in sentence:
    idx.append(word2index[word])
  return idx

def softmax(x):
  e_x = np.exp(x)
  return e_x / e_x.sum(axis=0)

In [11]:
np.random.seed(42)
embed_size = 10
embed = np.random.rand(len(vocab), embed_size)
recurrent = np.eye(embed_size)
start = np.zeros(embed_size)
decoder = np.random.rand(embed_size, len(vocab))
one_hot = np.eye(len(vocab))

In [12]:
def predict(sent):
  layers = list()
  layer = {}
  layer['hidden'] = start
  layers.append(layer)
  loss = 0
  preds = list()

  for target_i in range(len(sent)):
    layer = {}
    layer['pred'] = softmax(layers[-1]['hidden'].dot(decoder))
    layer['hidden'] = layers[-1]['hidden'].dot(recurrent) + embed[sent[target_i]]
    layers.append(layer)
  return layers, loss


In [15]:
alpha = 0.001
for iter in range(1000):
  sent = word2indices(tokens[iter % len(tokens)][1:])
  layers, loss = predict(sent)
  for layer_idx in reversed(range(len(layers))):
    layer = layers[layer_idx]
    target = sent[layer_idx - 1]
    if (layer_idx > 0):
      layer['output_delta'] = layer['pred'] - one_hot[target]
      new_hidden_delta = layer['output_delta'].dot(decoder.transpose())
      if (layer_idx == len(layers) - 1):
        layer['hidden_delta'] = new_hidden_delta
      else:
        layer['hidden_delta'] = new_hidden_delta + layers[layer_idx + 1]['hidden_delta'].dot(recurrent.transpose())
    else:
      layer['hidden_delta'] = layers[layer_idx+1]['hidden_delta'].dot(recurrent.transpose())

start -= layers[0]['hidden_delta'] * alpha / float(len(sent))
for layer_idx, layer in enumerate(layers[1:]):
  decoder -= np.dot(layers[layer_idx]['hidden'].reshape(-1, 1),layer['output_delta'].reshape(1, -1)) * alpha / float(len(sent))
  embed_idx = sent[layer_idx]
  embed[embed_idx] -= layers[layer_idx]['hidden_delta'] * alpha / float(len(sent))
  recurrent -= np.dot(layers[layer_idx]['hidden'].reshape(-1, 1),layer['hidden_delta'].reshape(1, -1)) * alpha / float(len(sent))

sent_index = 4
l, _ = predict(word2indices(tokens[sent_index]))
print(tokens[sent_index])
for i, each_layer in enumerate(l[1:-1]):
  input = tokens[sent_index][i]
  true = tokens[sent_index][i+1]
  pred = vocab[each_layer['pred'].argmax()]
  print("Prev Input:" + input + (" " * (12 - len(input))) + "True:" + true + (" " * (15 - len(true))) + "Pred:" + pred)


['sandra', 'moved', 'to', 'the', 'garden.']
Prev Input:sandra      True:moved          Pred:garden.
Prev Input:moved       True:to             Pred:	bathroom	14
Prev Input:to          True:the            Pred:	bathroom	7
Prev Input:the         True:garden.        Pred:	bathroom	7
